# Quick Start: Generating Traces with the ASQ Toolkit

This notebook shows how to generate traces with the ASQ toolkit in just a few steps: from setup, to executing trace-generation, to persisting traces to disk.

## Supported Agents and Generators

To see which agent families and generators are currently supported, you can use the following helper functions.

In [5]:
from trace_generation import help_supported_agents, help_supported_generators

help_supported_agents()
help_supported_generators('search_r1')
help_supported_generators('autorefine')

Supported agent families:
	search_r1
	autorefine
Agent search_r1 supports the following generators:
	Qwen-3B: PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-em-ppo-v0.3
	Qwen-7B: PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-7b-em-ppo
	Qwen-14B: PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-14b-it-em-ppo-v0.2
Agent autorefine supports the following generators:
	autorefine: yrshi/AutoRefine-Qwen2.5-3B-Instruct


# Generating ASQ Traces

To generate traces, run the command below. It will execute the pipeline, build the retrieval index if missing, and save outputs (`answers.tsv`, `iter_queries.tsv`, `retrieved_docs.tsv`, and optionally `thoughts.tsv`) under the configured data directory. Adjust the dataset, model, and retrieval parameters as needed.

In [ ]:
!python -m trace_generation.tutorial_main \
  --dataset hotpotqa-test \
  --model_type search_r1 \
  --model_id Qwen-3B \
  --agent_k 3 \
  --wmodel BM25 \
  --retriever_k 1000 \
  --rerank \
  --reranker_k 100 \
  --ram_limit 65000 \
  --extract_thoughts

INFO: Arguments: Namespace(dataset='hotpotqa-test', agent_k=3, model_type='search_r1', model_id='Qwen-3B', wmodel='BM25', retriever_k=1000, reranker_k=100, ram_limit=65000, max_queries=None, judged_queries_only=False, extract_thoughts=True, rerank=True)
INFO: Number of unique qids: 7405
Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
09:54:14.652 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1 GiB of memory would be required.
config.json: 100%|█████████████████████████████| 666/666 [00:00<00:00, 2.67MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 169kB/s]
vocab.txt: 232kB [00:00, 11.7MB/s]
tokenizer.json: 466kB [00:00, 11.7MB/s]
Loading weights: 1

## Generating Traces for Other IRDS Datasets

To generate traces for queries in other IRDS datasets using our setting, use `trace_generation.tutorial_main` and pass the dataset name **without** the `irds:` prefix. For example, to use `irds:antique/test`, set `dataset_name = "antique/test"`.

In [3]:
dataset_name = "vaswani" 
!python -m trace_generation.tutorial_main \
  --dataset {dataset_name} \
  --model_type search_r1 \
  --model_id Qwen-3B \
  --agent_k 3 \
  --wmodel BM25 \
  --retriever_k 1000 \
  --ram_limit 65000 \
  --extract_thoughts

INFO: Arguments: Namespace(dataset='vaswani', agent_k=3, model_type='search_r1', model_id='Qwen-3B', wmodel='BM25', retriever_k=1000, reranker_k=100, ram_limit=65000, max_queries=None, judged_queries_only=False, extract_thoughts=True, rerank=False)
INFO: Number of unique qids: 93
Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
INFO: Initialising search_r1 (will use top 3 documents).
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 435/435 [00:01<00:00, 324.25it/s, Materializing param=m
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
INFO: Starting search for 93 queries.
INFO: Outp

## Generating Traces Using Other Retrieval Pipelines

You can also generate ASQ traces with a custom retrieval pipeline instead of the default setup.
To do so, pass your own [PyTerrier transformer](https://pyterrier.readthedocs.io/en/latest/transformer.html#pyterrier.Indexer) when building the agent.

In the example below, we build a PyTerrier `Transformer` pipeline and pass it to the agent, then run trace generation as usual.

In [4]:
import pyterrier as pt
from sentence_transformers import CrossEncoder
from data_utils.retrieval_utils import calc_index_fpath, index_lexical
from trace_generation.rag_agent.agent import build_agent
import pandas as pd

dataset_name = "hotpotqa-test"
index_path = calc_index_fpath(dataset_name)
index_lexical(index_path, dataset_name) # index

# few queries (quick test)
queries_df = pd.DataFrame()
queries_df['qid'] = ["query_1", "query_2", "query_3"]
queries_df['query'] = ["Why does increasing atmospheric CO₂ both enhance plant growth in controlled experiments and yet correlate with declining global crop yields in many regions?", "Why does the Moon always show the same face to Earth, and how does this relate to tidal forces and the Moon’s orbital period?", "Why does increasing altitude lead to lower oxygen availability for humans, and how do changes in atmospheric pressure and gas diffusion explain this effect?"]

# build retrieval pipeline with PyTerrier (TF-IDF + MiniLM reranker)
retriever = pt.terrier.Retriever(index_path, wmodel="TF_IDF", metadata=["docno", "text"]) # retriever
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2") # reranker
reranker = pt.apply.doc_score(lambda row: ce.predict([(row["query"], row["text"])])[0])

retrieval_pipeline = retriever % 1000 >> reranker % 3

# build the agent
agent = build_agent(
    model_type="search_r1",
    model_id="Qwen-3B",
    retrieval_pipeline=retrieval_pipeline,
    agent_k=3,
 )

# quick test with 3 queries
sample_queries = queries_df.head(3)[["qid", "query"]]

res = agent(sample_queries)
res

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


10:03:49.668 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1 GiB of memory would be required.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


10:04:01.260 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1 GiB of memory would be required.


,qid,query,answer,output,iteration,all_queries,all_docs,all_docids,all_thoughts
0,query_1,Why does increasing atmospheric CO₂ both enhan...,Climate change,<|im_start|>system\nYou are a helpful assistan...,0,[],[],[],"[(0, So, while increasing atmospheric CO₂ can ..."
1,query_2,Why does the Moon always show the same face to...,Tidal locking,<|im_start|>system\nYou are a helpful assistan...,0,[],[],[],"[(0, Now I have found the answer to the questi..."
2,query_3,Why does increasing altitude lead to lower oxy...,Decreased atmospheric pressure,<|im_start|>system\nYou are a helpful assistan...,0,[],[],[],"[(0, Now that I know gas diffusion is the proc..."


### Inspecting the Resulting DataFrame

The resulting DataFrame has one row per query, and its columns include:
- `qid`, `query`: identifiers and original queries.
- `answer`: final answer extracted from the agent output.
- `output`: full model output text.
- `iteration`: number of reasoning iterations performed.
- `all_queries`: list of `(iteration, llm_query)` tuples.
- `all_docids`: list of `(iteration, [docid, ...])` tuples.
- `all_thoughts`: list of `(iteration, thought)` tuples (if thoughts are enabled).

To extract per-iteration data, explode these list columns or process the DataFrame using our `DataFrameExtractor`class as explained later on. 

# Persisting Trace Artifacts to Disk
By default, the agent logs outputs and returns them into a DataFrame, but it does not persist these artifacts to disk.

In [6]:
res = agent(sample_queries)
res

,qid,query,answer,output,iteration,all_queries,all_docs,all_docids,all_thoughts
0,query_1,Why does increasing atmospheric CO₂ both enhan...,Negative effects of increasing atmospheric CO₂...,<|im_start|>system\nYou are a helpful assistan...,0,[],[],[],"[(0, So, the correlation between increasing at..."
1,query_2,Why does the Moon always show the same face to...,Tidal locking,<|im_start|>system\nYou are a helpful assistan...,0,[],[],[],"[(0, Now I understand that the Moon's orbital ..."
2,query_3,Why does increasing altitude lead to lower oxy...,Decreased atmospheric pressure and gas diffusion,<|im_start|>system\nYou are a helpful assistan...,0,[],[],[],"[(0, Based on the information found, I can now..."


To log data and store them to disk using our format, use `build_output_paths` and `DataFrameExtractor` as shown below. This will write the TSV artifacts (`answers.tsv`, `iter_queries.tsv`, `retrieved_docs.tsv`, and optionally `thoughts.tsv`).

In [7]:
from trace_generation.output_paths import build_output_paths
from trace_generation.extraction.extractor_df import DataFrameExtractor

# save trace artifacts to disk
output_paths = build_output_paths(
    data_dir="./data/tutorial_generation",
    dataset=dataset_name,
    retriever="TF_IDF",
    retriever_k=1000,
    agent_family="search_r1",
    agent_generator="Qwen-3B",
    reranker="minilm",
    reranker_k=3,
 )

extractor = DataFrameExtractor(verbose=True)
_ = extractor.extract_all_and_save(res, output_paths, verbose=True)
print("Saved to:", output_paths.base_dir)

Initialized DataFrameExtractor with extraction order: ['answers', 'thoughts', 'iter_queries', 'retrieved_docs']
Extracting answers...
Extracting thoughts...
Extracting iter_queries...
Extracting retrieved_docs...
Error extracting retrieved_docs: No retrieved documents found in ranked list DataFrame.
Saved to: data/tutorial_generation/traces/hotpotqa-test/TF_IDF_k1000_minilm_k3/search_r1/Qwen-3B


To log data and store them to disk using custom paths, instantiate a `trace_generation.output_paths.OutputPaths` object.

In [8]:

from pathlib import Path
from trace_generation.output_paths import OutputPaths
from trace_generation.extraction.extractor_df import DataFrameExtractor

custom_base = Path("./custom_traces") / dataset_name / "TF_IDF_k1000_minilm_k3" / "search_r1" / "Qwen-3B"
custom_paths = OutputPaths(
    base_dir=custom_base,
    answers=custom_base / "custom_answers.tsv",
    iter_queries=custom_base / "custom_synthetic_queries.tsv",
    retrieved_docs=custom_base / "custom_ranked_lists.tsv",
    thoughts=custom_base / "custom_thoughts.tsv",
 )

custom_extractor = DataFrameExtractor(verbose=True)
_ = custom_extractor.extract_all_and_save(res, custom_paths, verbose=True)
print("Saved to:", custom_paths.base_dir)

Initialized DataFrameExtractor with extraction order: ['answers', 'thoughts', 'iter_queries', 'retrieved_docs']
Extracting answers...
Extracting thoughts...
Extracting iter_queries...
Extracting retrieved_docs...
Error extracting retrieved_docs: No retrieved documents found in ranked list DataFrame.
Saved to: custom_traces/hotpotqa-test/TF_IDF_k1000_minilm_k3/search_r1/Qwen-3B
